In [1]:
import pandas as pd
import pickle
import openpyxl
import numpy as np
EXCEL_FILE = "formatted_results_9-18.xlsx"

def export_excel_from_checkpoint(checkpoint_file):

    # Load from pickle
    with open(checkpoint_file, 'rb') as f:
        ck = pickle.load(f)
    all_results = ck['all_results']

    model_map = {
        'SVM': 'SVM', 'kNN': 'KNN',
        'Decision Tree': 'DT', 'Random Forest': 'RF', 'MLP': 'MLP'
    }

    def build_sheet(phase):
        all_rows = []
        for i, ds in enumerate(sorted(all_results.keys()), 1):
            block = all_results[ds][phase]
            cols  = []
            for m in model_map.values():
                cols += [f'{m}-Accuracy', f'{m}-F1']
            cols += [f'{m} Parameters' for m in model_map.values()]

            all_rows.append([f'Data {i}'] + [''] * len(cols))
            all_rows.append([''] + cols)

            n_folds   = len(next(iter(block.values()))['acc'])
            fold_data = []
            for fold in range(n_folds):
                row, num_row = [f'Fold {fold+1}'], []
                for model in model_map:
                    acc = block[model]['acc'][fold]
                    f1  = block[model]['f1'][fold]
                    row += [round(acc, 4), round(f1, 4)]
                    num_row += [acc, f1]
                for model in model_map:
                    row.append(str(block[model]['params'][fold]))
                all_rows.append(row)
                fold_data.append(num_row)

            arr      = np.array(fold_data)
            means    = np.mean(arr, axis=0)
            stds     = np.std(arr,  axis=0)
            mean_std = [f'{m:.4f} ± {s:.4f}' for m, s in zip(means, stds)]
            all_rows.append(['Mean ± Std'] + mean_std + [''] * len(model_map))
            all_rows.append([''] * (len(cols) + 1))

        return pd.DataFrame(all_rows)

    with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
        build_sheet('baseline').to_excel(
            writer, sheet_name='Baseline', index=False, header=False)
        build_sheet('relieff').to_excel(
            writer, sheet_name='ReliefF',  index=False, header=False)

    print(f"✅  Excel saved → '{EXCEL_FILE}'")

# Run it
export_excel_from_checkpoint('results_checkpoint2.pkl')

✅  Excel saved → 'formatted_results_9-18.xlsx'
